In [1]:
from pathlib import Path
import json

ROOT = Path('/Users/luyiming/Projects/edinet-agentic')
BASE = ROOT / 'reproduction/outputs'
EXPERIMENTS = ['EXP-R-0002', 'EXP-R-0003']

target_files = []
for exp in EXPERIMENTS:
    target_files.extend(sorted((BASE / exp).glob('*/results.jsonl')))

target_files


[PosixPath('/Users/luyiming/Projects/edinet-agentic/reproduction/outputs/EXP-R-0002/claude-haiku-4-5-20251001/results.jsonl'),
 PosixPath('/Users/luyiming/Projects/edinet-agentic/reproduction/outputs/EXP-R-0002/o4-mini-2025-04-16/results.jsonl'),
 PosixPath('/Users/luyiming/Projects/edinet-agentic/reproduction/outputs/EXP-R-0003/claude-haiku-4-5-20251001/results.jsonl'),
 PosixPath('/Users/luyiming/Projects/edinet-agentic/reproduction/outputs/EXP-R-0003/o4-mini-2025-04-16/results.jsonl')]

In [2]:
def predicted_positive_rate(results_path: Path):
    n = 0
    predicted_positive = 0

    with results_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            rec = json.loads(line)
            n += 1
            predicted_positive += int(rec['prediction'] == 1)

    if n == 0:
        raise ValueError(f'Empty file: {results_path}')

    rate = predicted_positive / n
    return rate, predicted_positive, n

rows = []
for path in target_files:
    exp = path.parents[1].name
    model = path.parent.name
    rate, pos, n = predicted_positive_rate(path)
    rows.append((exp, model, rate, pos, n))

rows = sorted(rows, key=lambda x: (x[0], x[1]))
rows


[('EXP-R-0002', 'claude-haiku-4-5-20251001', 0.96, 48, 50),
 ('EXP-R-0002', 'o4-mini-2025-04-16', 0.06, 3, 50),
 ('EXP-R-0003', 'claude-haiku-4-5-20251001', 0.36, 18, 50),
 ('EXP-R-0003', 'o4-mini-2025-04-16', 0.02, 1, 50)]

In [3]:
for exp, model, rate, pos, n in rows:
    print(f'{model} @ {exp}: pi_hat_plus = {pos}/{n} = {rate:g}')


claude-haiku-4-5-20251001 @ EXP-R-0002: pi_hat_plus = 48/50 = 0.96
o4-mini-2025-04-16 @ EXP-R-0002: pi_hat_plus = 3/50 = 0.06
claude-haiku-4-5-20251001 @ EXP-R-0003: pi_hat_plus = 18/50 = 0.36
o4-mini-2025-04-16 @ EXP-R-0003: pi_hat_plus = 1/50 = 0.02


In [4]:
def confusion_counts(results_path: Path):
    tp = fp = tn = fn = 0

    with results_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            rec = json.loads(line)
            y_true = rec['label']
            y_pred = rec['prediction']

            if y_true == 1 and y_pred == 1:
                tp += 1
            elif y_true == 0 and y_pred == 1:
                fp += 1
            elif y_true == 0 and y_pred == 0:
                tn += 1
            elif y_true == 1 and y_pred == 0:
                fn += 1

    return tp, fp, tn, fn

conf_rows = []
for path in target_files:
    exp = path.parents[1].name
    model = path.parent.name
    tp, fp, tn, fn = confusion_counts(path)
    conf_rows.append((exp, model, tp, fp, tn, fn))

conf_rows = sorted(conf_rows, key=lambda x: (x[0], x[1]))
conf_rows


[('EXP-R-0002', 'claude-haiku-4-5-20251001', 27, 21, 2, 0),
 ('EXP-R-0002', 'o4-mini-2025-04-16', 2, 1, 22, 25),
 ('EXP-R-0003', 'claude-haiku-4-5-20251001', 9, 9, 14, 18),
 ('EXP-R-0003', 'o4-mini-2025-04-16', 1, 0, 23, 26)]

In [5]:
for exp, model, tp, fp, tn, fn in conf_rows:
    print(f'{model} @ {exp}: TP={tp}, FP={fp}, TN={tn}, FN={fn}')


claude-haiku-4-5-20251001 @ EXP-R-0002: TP=27, FP=21, TN=2, FN=0
o4-mini-2025-04-16 @ EXP-R-0002: TP=2, FP=1, TN=22, FN=25
claude-haiku-4-5-20251001 @ EXP-R-0003: TP=9, FP=9, TN=14, FN=18
o4-mini-2025-04-16 @ EXP-R-0003: TP=1, FP=0, TN=23, FN=26


In [6]:
from collections import OrderedDict

def load_rows_by_doc_id(results_path: Path):
    rows = OrderedDict()
    with results_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            rows[rec['doc_id']] = rec
    return rows


In [7]:
MAX_ROWS_TO_SHOW = 3

for exp in EXPERIMENTS:
    exp_files = sorted((BASE / exp).glob('*/results.jsonl'))
    if len(exp_files) != 2:
        raise ValueError(f'Expected exactly 2 model files under {exp}, got {len(exp_files)}')

    model_a, model_b = exp_files[0].parent.name, exp_files[1].parent.name
    rows_a = load_rows_by_doc_id(exp_files[0])
    rows_b = load_rows_by_doc_id(exp_files[1])

    common_doc_ids = sorted(set(rows_a) & set(rows_b))
    if not common_doc_ids:
        print(f'===== {exp} =====')
        print('No overlapping doc_id between the two model outputs.')
        print()
        continue

    diffs = []
    for doc_id in common_doc_ids:
        pa = float(rows_a[doc_id]['prob'])
        pb = float(rows_b[doc_id]['prob'])
        abs_diff = abs(pa - pb)
        diffs.append((abs_diff, doc_id, pa, pb))

    max_diff = max(x[0] for x in diffs)
    winners = [x for x in diffs if x[0] == max_diff][:MAX_ROWS_TO_SHOW]
    total_ties = sum(1 for x in diffs if x[0] == max_diff)

    print(f'===== {exp} =====')
    print(f'models: {model_a} vs {model_b}')
    print(f'max_abs_prob_diff = {max_diff:g}')
    if total_ties > MAX_ROWS_TO_SHOW:
        print(f'ties at max diff: {total_ties} (showing first {MAX_ROWS_TO_SHOW})')

    for i, (abs_diff, doc_id, pa, pb) in enumerate(winners, 1):
        print(f'-- case {i}: doc_id={doc_id}, abs_diff={abs_diff:g} ({model_a}: {pa:g}, {model_b}: {pb:g})')
        print(f'{model_a} row:')
        print(json.dumps(rows_a[doc_id], ensure_ascii=False, indent=2))
        print(f'{model_b} row:')
        print(json.dumps(rows_b[doc_id], ensure_ascii=False, indent=2))

    print()


===== EXP-R-0002 =====
models: claude-haiku-4-5-20251001 vs o4-mini-2025-04-16
max_abs_prob_diff = 0.67
-- case 1: doc_id=S1009C5F, abs_diff=0.67 (claude-haiku-4-5-20251001: 0.72, o4-mini-2025-04-16: 0.05)
claude-haiku-4-5-20251001 row:
{
  "doc_id": "S1009C5F",
  "edinet_code": "E02030",
  "label": 0,
  "prediction": 1,
  "prob": 0.72,
  "reasoning": "Analysis reveals several significant red flags warranting fraud investigation: (1) DRAMATIC REVENUE DECLINE: Sales dropped 19.9% from ¥32.5B to ¥26.0B, yet SG&A expenses increased 9.8% (¥7.38B to ¥8.11B), suggesting management failed to adjust cost structure proportionally - unusual for legitimate operational contraction. (2) PROFIT COLLAPSE DISPROPORTIONATE TO REVENUE: Operating profit plummeted 73.9% (¥6.14B to ¥1.60B) while gross margin deteriorated from 41.6% to 37.3%, indicating either severe pricing pressure, cost inflation, or potential inventory/revenue recognition issues. (3) CASH FLOW DETERIORATION: Operating cash flow crashed 

In [8]:
MAX_ROWS_TO_SHOW = 3
exp_a, exp_b = 'EXP-R-0002', 'EXP-R-0003'

models_a = sorted(p.name for p in (BASE / exp_a).iterdir() if p.is_dir())
models_b = sorted(p.name for p in (BASE / exp_b).iterdir() if p.is_dir())
common_models = sorted(set(models_a) & set(models_b))

cross_exp_winners = []
for model in common_models:
    path_a = BASE / exp_a / model / 'results.jsonl'
    path_b = BASE / exp_b / model / 'results.jsonl'

    rows_a = load_rows_by_doc_id(path_a)
    rows_b = load_rows_by_doc_id(path_b)

    common_doc_ids = sorted(set(rows_a) & set(rows_b))
    diffs = []
    for doc_id in common_doc_ids:
        pa = float(rows_a[doc_id]['prob'])
        pb = float(rows_b[doc_id]['prob'])
        abs_diff = abs(pa - pb)
        diffs.append((abs_diff, doc_id, pa, pb))

    max_diff = max(x[0] for x in diffs)
    winners = [x for x in diffs if x[0] == max_diff]

    cross_exp_winners.append({
        'model': model,
        'exp_a': exp_a,
        'exp_b': exp_b,
        'rows_a': rows_a,
        'rows_b': rows_b,
        'max_diff': max_diff,
        'winners': winners[:MAX_ROWS_TO_SHOW],
        'total_ties': len(winners),
    })

cross_exp_winners


[{'model': 'claude-haiku-4-5-20251001',
  'exp_a': 'EXP-R-0002',
  'exp_b': 'EXP-R-0003',
  'rows_a': OrderedDict([('S100LQX5',
                {'doc_id': 'S100LQX5',
                 'edinet_code': 'E00032',
                 'label': 0,
                 'prediction': 1,
                 'prob': 0.72,
                 'reasoning': "Analysis reveals several significant red flags warranting fraud investigation: (1) DIVERGENT PROFIT TRENDS: Net income surged 68.5% (¥4.43B→¥7.47B) while operating profit declined 35.6% (¥8.69B→¥5.59B), creating an unusual disconnect where non-operating activities drove profitability despite operational deterioration. (2) SUSPICIOUS SPECIAL GAINS: Investment securities sales gains jumped dramatically from ¥20M to ¥4.08B (204x increase), representing 61% of pre-tax profit. This extraordinary one-time gain appears to be the primary driver of net income growth, raising questions about timing and necessity. (3) ACCOUNTING ANOMALY: Comprehensive income shows ¥18.

In [9]:
for item in cross_exp_winners:
    model = item['model']
    exp_a = item['exp_a']
    exp_b = item['exp_b']
    rows_a = item['rows_a']
    rows_b = item['rows_b']
    max_diff = item['max_diff']
    winners = item['winners']
    total_ties = item['total_ties']

    print(f'===== {model} =====')
    print(f'experiments: {exp_a} vs {exp_b}')
    print(f'max_abs_prob_diff = {max_diff:g}')
    if total_ties > MAX_ROWS_TO_SHOW:
        print(f'ties at max diff: {total_ties} (showing first {MAX_ROWS_TO_SHOW})')

    for i, (abs_diff, doc_id, pa, pb) in enumerate(winners, 1):
        print(f'-- case {i}: doc_id={doc_id}, abs_diff={abs_diff:g} ({exp_a}: {pa:g}, {exp_b}: {pb:g})')
        print(f'{exp_a} row:')
        print(json.dumps(rows_a[doc_id], ensure_ascii=False, indent=2))
        print(f'{exp_b} row:')
        print(json.dumps(rows_b[doc_id], ensure_ascii=False, indent=2))
        print()

    print()


===== claude-haiku-4-5-20251001 =====
experiments: EXP-R-0002 vs EXP-R-0003
max_abs_prob_diff = 0.44
-- case 1: doc_id=S100ET3W, abs_diff=0.44 (EXP-R-0002: 0.72, EXP-R-0003: 0.28)
EXP-R-0002 row:
{
  "doc_id": "S100ET3W",
  "edinet_code": "E03314",
  "label": 1,
  "prediction": 1,
  "prob": 0.72,
  "reasoning": "Analysis reveals several concerning patterns that warrant fraud investigation: (1) EXTREME VALUATION ANOMALY: P/E ratio exploded from 188.8 (Prior2Year) to 1790.6 (CurrentYear) despite minimal earnings recovery (net income only 10.8M yen), suggesting potential earnings manipulation or stock price inflation. (2) EQUITY STRUCTURE INCONSISTENCY: Pure equity increased 57.8% (5.5B to 8.7B yen) while net income remained negligible, driven primarily by capital increases (959M yen) and treasury stock reduction (978M yen) rather than operational performance. This suggests potential equity restructuring to mask poor fundamentals. (3) ASSET QUALITY CONCERNS: Total assets grew 7.8% while o